## CDC_PROCESS Python Variant

Hieronder vind je de Python variant van de het CDC proces.


### Imports

In [136]:
import snowflake.connector
from snowflake.connector import DictCursor
from typing import Any, Optional
import os

### Connectie met Snowflake

In [137]:
print("❄ Verbinding maken met Snowflake...")
# Connectie parameters van Snowflake en de database
try:
    conn = snowflake.connector.connect(
        user="williamvdm",
        password=os.getenv("SNOWFLAKE_PASSWORD"),
        account="rgqgoxq-nwb12951",
        warehouse="COMPUTE_WH",
        database="CDC_PYTHON_DB",
        role="SYSADMIN"
    )
    print("❄ Verbinding met Snowflake: geslaagd")
except Exception as e:
    print(f"❄ Verbinding met Snowflake: mislukt, error:{e}")



❄ Verbinding maken met Snowflake...
❄ Verbinding met Snowflake: geslaagd


### Initaliseren
- Config ophalen
- Business kolommen ophalen

In [138]:


from snowflake.connector import DictCursor

# Functie om het CDC proces te initialiseren met de config, business kolommen en ROW_HASH toevoegen/genereren
def initialise_cdc(config_id: int, v_runid: Any) -> dict[str, Any] | str: # PEP 484 type hinting bewijs
    with conn.cursor(DictCursor) as cursor:
        print(f"❄ Initialiseren CDC proces voor CONFIG_ID: {config_id} en RUN_ID: {v_runid}...")
        # Config van de entiteit ophalen
        cursor.execute(
            """
            SELECT ENTITY_NAME, PRIMARY_KEY_COLUMN, SOURCE_TABLE, TARGET_TABLE, DELETE_STRATEGY, ERROR_STRATEGY, UPDATE_STRATEGY
            FROM CDC.CDC_CONFIG
            WHERE CONFIG_ID = %s
              AND IS_ACTIVE = TRUE
            """,
            (config_id,),
        )
        config = cursor.fetchone()

        v_entity = config["ENTITY_NAME"] 
        v_pk = config["PRIMARY_KEY_COLUMN"]
        v_source = config["SOURCE_TABLE"]
        v_target = config["TARGET_TABLE"]
        v_delete_strategy = config["DELETE_STRATEGY"]
        v_error_strategy = config["ERROR_STRATEGY"]
        v_update_strategy = config["UPDATE_STRATEGY"]

        print(f"❄ Config opgehaald: ENTITY_NAME={v_entity}, PRIMARY_KEY_COLUMN={v_pk}, SOURCE_TABLE={v_source}, TARGET_TABLE={v_target}, DELETE_STRATEGY={v_delete_strategy}, ERROR_STRATEGY={v_error_strategy}, UPDATE_STRATEGY={v_update_strategy}")
        
        print(f"❄ Business kolommen van de entiteit ophalen...")
        # Business kolommen van de entiteit ophalen (alle kolommen behalve kolommen die we gebruiken voor CDC logica: ROW_HASH, START_TS, END_TS, IS_ACTIVE, CDC_OPERATION en de primary key).
        cursor.execute(
            """
            SELECT LISTAGG('"' || COLUMN_NAME || '"', ', ') AS BUSINESS_COLUMNS
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_SCHEMA = 'STAGING'
              AND TABLE_NAME = UPPER(SPLIT_PART(%s, '.', -1))
              AND COLUMN_NAME NOT IN ('ROW_HASH', 'START_TS', 'END_TS', 'IS_ACTIVE', 'CDC_OPERATION')
              AND COLUMN_NAME != %s
            """,
            (v_source, v_pk,),
        )
        row = cursor.fetchone()
        v_business_columns = row["BUSINESS_COLUMNS"]

        print(f"❄ Business kolommen opgehaald: {v_business_columns}")

        # Business kolommen voor de update statement (zelfde als hierboven maar dan specifiek voor de update statement: "COLUMN_NAME = s.COLUMN_NAME").
        #     """
        #     SELECT LISTAGG('"' || COLUMN_NAME || '" = s."' || COLUMN_NAME || '"', ', ') AS BUSINESS_COLUMNS_UPDATE
        #     FROM INFORMATION_SCHEMA.COLUMNS
        #     WHERE TABLE_SCHEMA = 'STAGING'
        #       AND TABLE_NAME = UPPER(SPLIT_PART(:v_source, '.', -1))
        #       AND COLUMN_NAME NOT IN ('ROW_HASH', 'START_TS', 'END_TS', 'IS_ACTIVE', 'CDC_OPERATION')
        #       AND COLUMN_NAME != {v_pk}
        #     """,
        # )
        # row = cursor.fetchone()
        # v_business_columns_update = row["BUSINESS_COLUMNS_UPDATE"]

        print(f"❄ ROW_HASH kolom toevoegen aan {v_source}...")

        # ROW_HASH kolom toevoegen en genereren en vullen indien nodig
        cursor.execute(f"ALTER TABLE {v_source} ADD COLUMN IF NOT EXISTS ROW_HASH STRING")

        print(f"❄ ROW_HASH kolom (al) toegevoegd aan {v_source}")
        print(f"❄ ROW_HASH genereren en vullen voor alle records in {v_source}...")
        cursor.execute(
            f"""
            UPDATE {v_source}
            SET ROW_HASH = SHA2(TO_VARCHAR(OBJECT_CONSTRUCT(* EXCLUDE ROW_HASH)), 256)
            """
        )
        conn.commit()

        print(f"❄ ROW_HASH gegenereerd voor alle records in {v_source}.")

    return {
        "v_entity": v_entity,
        "v_pk": v_pk,
        "v_source": v_source,
        "v_target": v_target,
        "v_delete_strategy": v_delete_strategy,
        "v_error_strategy": v_error_strategy,
        "v_update_strategy": v_update_strategy,
        "v_business_columns": v_business_columns,
    }

### Errors detecteren & loggen

In [ ]:
# Functie om alle fouteen als duplicaten en lege/null pk's te detecteren en loggen
def detect_errors(v_runid: int, init_values: dict -> dict[str, Any]) -> dict[str, int] | str: # PEP 484 type hinting bewijs
    v_entity = init_values["v_entity"]
    v_pk = init_values["v_pk"]
    v_source = init_values["v_source"]

    print(f"❄ Beginnen met verwerken van entity {v_entity}...")

    with conn.cursor(DictCursor) as cursor:
        print(f"❄ Duplicate inserts detecteren...")

        # Duplicate inserts detecteren: zelfde pk en zelfde hash
        v_sql = f"""
        INSERT INTO LOGGING.RUN_ERROR_LOG (RUN_ID, ENTITY_NAME, ERROR_CODE, ERROR_ROW)
        SELECT {v_runid}, '{v_entity}', 'DUPLICATE_INSERT', OBJECT_CONSTRUCT(*)
        FROM (
            SELECT s.*, COUNT(*) OVER (PARTITION BY s.{v_pk}, s.ROW_HASH) AS CNT
            FROM {v_source} s
            WHERE s.{v_pk} IS NOT NULL
              AND s.{v_pk} != ''
              AND s.ROW_HASH IS NOT NULL
        ) t
        WHERE t.CNT > 1
        """
        cursor.execute(v_sql)
        v_duplicate_inserts = cursor.rowcount

        print(f"❄ Duplicate inserts gedetecteerd: {v_duplicate_inserts}")
        print(f"❄ Duplicate updates detecteren...")

        # Duplicate updates: zelfde pk en verschillende hash
        v_sql = f"""
        INSERT INTO LOGGING.RUN_ERROR_LOG (RUN_ID, ENTITY_NAME, ERROR_CODE, ERROR_ROW)
        SELECT {v_runid}, '{v_entity}', 'DUPLICATE_UPDATE', OBJECT_CONSTRUCT(*)
        FROM (
            SELECT s.*
            FROM {v_source} s
            WHERE s.{v_pk} IS NOT NULL
              AND s.{v_pk} != ''
              AND s.ROW_HASH IS NOT NULL
              AND EXISTS (
                  SELECT 1
                  FROM {v_source} s2
                  WHERE s2.{v_pk} = s.{v_pk}
                    AND s2.ROW_HASH IS NOT NULL
                    AND s2.ROW_HASH <> s.ROW_HASH
              )
        ) t
        """
        cursor.execute(v_sql)
        v_duplicate_updates = cursor.rowcount

        print(f"❄ Duplicate updates gedetecteerd: {v_duplicate_updates}")
        print(f"❄ Key errors detecteren...")

        # # Ongewijzigde rijen: zelfde pk en zelfde hash in target (en maar 1 record in source met die pk)
        # v_sql = f"""
        # SELECT COUNT(*)
        # FROM {v_target} t
        # JOIN {v_source} s
        # ON s.{v_pk} = t.{v_pk}
        # WHERE t.IS_ACTIVE = TRUE
        #     AND s.{v_pk} IS NOT NULL
        #     AND s.{v_pk} != ''
        #     AND s.ROW_HASH = t.ROW_HASH
        #     AND (SELECT COUNT(*) FROM {v_source} s2 WHERE s2.{v_pk} = s.{v_pk}) = 1
        # """
        # cursor.execute(v_sql)
        # v_unchanged = cursor.rowcount

        # Key errors: NULL of lege pk
        v_sql = f"""
        INSERT INTO LOGGING.RUN_ERROR_LOG (RUN_ID, ENTITY_NAME, ERROR_CODE, ERROR_ROW)
        SELECT {v_runid}, '{v_entity}', 'PRIMARY_KEY_ERROR', OBJECT_CONSTRUCT(*)
        FROM (
            SELECT s.*
            FROM {v_source} s
            WHERE s.{v_pk} IS NULL
               OR s.{v_pk} = ''
        )
        """
        cursor.execute(v_sql)
        v_key_errors = cursor.rowcount

        print(f"❄ Key errors gedetecteerd: {v_key_errors}")

    conn.commit()

    return {
        "v_duplicate_inserts": v_duplicate_inserts,
        "v_duplicate_updates": v_duplicate_updates,
        # "v_unchanged": v_unchanged,
        "v_key_errors": v_key_errors,
    }

### Inserts uitvoeren

In [ ]:
# Functie om de inserts uit te voeren (alle rijen die geen fouten bevatten en nog niet in target staan)
def execute_inserts(v_runid: Any, init_values: dict -> dict[str, Any]) -> int | str: # PEP 484 type hinting bewijs
    v_target = init_values["v_target"]
    v_pk = init_values["v_pk"]
    v_source = init_values["v_source"]
    v_business_columns = init_values["v_business_columns"]

    target_cols = f"ROW_HASH, START_TS, IS_ACTIVE, CDC_OPERATION, {v_pk}" + (f", {v_business_columns}")
    stage_cols = ("s.ROW_HASH, CURRENT_TIMESTAMP(), TRUE, 'I', s." + v_pk + (f", s.{v_business_columns.replace(', ', ', s.')}"))

    print(f"❄ Inserts uitvoeren...")

    v_sql = f"""
    INSERT INTO {v_target} ({target_cols})
    SELECT {stage_cols}
    FROM {v_source} s
    LEFT JOIN {v_target} t
      ON t.{v_pk} = s.{v_pk} AND t.IS_ACTIVE = TRUE
    WHERE t.{v_pk} IS NULL
      AND s.{v_pk} IS NOT NULL
      AND s.{v_pk} != ''
      AND (SELECT COUNT(*) FROM {v_source} s2 WHERE s2.{v_pk} = s.{v_pk}) = 1
    """

    with conn.cursor() as cursor:
        cursor.execute(v_sql)
        v_inserts = cursor.rowcount

    print(f"❄ Inserts uitgevoerd. Aantal inserts: {v_inserts}")
        
    conn.commit()
    return v_inserts

### Updates uitvoeren

In [ ]:
# Functie om de updates uit te voeren (alle rijen die geen fouten bevatten en al in target staan)
def execute_updates(v_runid: Any, init_values: dict -> dict[str, Any]) -> int | str: # PEP 484 type hinting bewijs
    v_target = init_values["v_target"]
    v_pk = init_values["v_pk"]
    v_source = init_values["v_source"]
    v_business_columns = init_values["v_business_columns"]
    v_update_strategy = (init_values["v_update_strategy"]).upper()

    target_cols = f"ROW_HASH, START_TS, IS_ACTIVE, CDC_OPERATION, {v_pk}" + (f", {v_business_columns}")
    stage_cols = ("s.ROW_HASH, CURRENT_TIMESTAMP(), TRUE, 'U', s." + v_pk + (f", s.{v_business_columns.replace(', ', ', s.')}"))
    update_cols = ", ".join(f'"{colname.strip()}" = s."{colname.strip()}"' for colname in v_business_columns.split(",") if colname.strip())

    print(f"❄ Updates uitvoeren...")

    with conn.cursor() as cursor:
        # HISTORY: oude actieve versie afsluiten en nieuwe versie toevoegen, zodat historie beschikbaar blijft
        if v_update_strategy == "HISTORY":
            # Oude actieve versie afsluiten
            v_sql = f"""
            UPDATE {v_target} t
            SET IS_ACTIVE = FALSE, END_TS = CURRENT_TIMESTAMP()
            WHERE t.IS_ACTIVE = TRUE
                AND EXISTS (
                    SELECT 1
                    FROM {v_source} s
                    WHERE s.{v_pk} = t.{v_pk}
                    AND s.{v_pk} IS NOT NULL
                    AND s.{v_pk} != ''
                    AND s.ROW_HASH <> t.ROW_HASH
                    AND (
                        SELECT COUNT(*)
                        FROM {v_source} s2
                        WHERE s2.{v_pk} = s.{v_pk}
                    ) = 1
                )
            """
            cursor.execute(v_sql)

            # Nieuwe versie toevoegen
            v_sql = f"""
            INSERT INTO {v_target} ({target_cols})
            SELECT {stage_cols}
            FROM {v_source} s
            WHERE s.{v_pk} IS NOT NULL
                AND s.{v_pk} != ''
                AND (
                    SELECT COUNT(*)
                    FROM {v_source} s2
                    WHERE s2.{v_pk} = s.{v_pk}
                ) = 1
                AND NOT EXISTS (
                    SELECT 1
                    FROM {v_target} t
                    WHERE t.{v_pk} = s.{v_pk}
                    AND t.IS_ACTIVE = TRUE
                    AND t.ROW_HASH = s.ROW_HASH
                )
            """
        else:
            # OVERWRITE: actieve rij overschrijven en geen historie bijhouden
            v_sql = f"""
            UPDATE {v_target} t
            SET (ROW_HASH = s.ROW_HASH, START_TS = CURRENT_TIMESTAMP(), IS_ACTIVE = TRUE, CDC_OPERATION = 'U', {v_pk} = s.{v_pk}{f", {update_cols}"})
            FROM {v_source} s
            WHERE t.{v_pk} = s.{v_pk}
                AND s.{v_pk} IS NOT NULL
                AND s.{v_pk} != ''
                AND t.IS_ACTIVE = TRUE
                AND t.ROW_HASH <> s.ROW_HASH
                AND (
                    SELECT COUNT(*)
                    FROM {v_source} s2
                    WHERE s2.{v_pk} = s.{v_pk}
                ) = 1
            """

        cursor.execute(v_sql)
        v_updates = cursor.rowcount

        print(f"❄ Updates uitgevoerd met strategie {v_update_strategy}. Aantal updates: {v_updates}")

    conn.commit()
    return v_updates

### Deletes uitvoeren

In [ ]:
# Functie om de deletes uit te voeren (alle rijen die geen fouten bevatten en niet meer in source staan)
# We gaan er hier van uit dat elke keer de volledige dataset in staging staat, dus dat we kunnen vergelijken op basis van het ontbreken van de pk in source.
def execute_deletes(v_runid: Any, init_values: dict -> dict[str, Any]) -> int | str: # PEP 484 type hinting bewijs
    v_target = init_values["v_target"]
    v_source = init_values["v_source"]
    v_pk = init_values["v_pk"]
    v_delete_strategy = (init_values["v_delete_strategy"] or "").upper()

    print(f"❄ Deletes uitvoeren...")

    # Soft delete strategie
    if v_delete_strategy == "SOFT":
        v_sql = f"""
        UPDATE {v_target} t
        SET IS_ACTIVE = FALSE, END_TS = CURRENT_TIMESTAMP(), CDC_OPERATION = 'D'
        WHERE t.IS_ACTIVE = TRUE
          AND NOT EXISTS (
            SELECT 1
            FROM {v_source} s
            WHERE s.{v_pk} = t.{v_pk}
          )
        """
    # Hard delete strategie    
    else:
        v_sql = f"""
        DELETE FROM {v_target} t
        WHERE t.IS_ACTIVE = TRUE
          AND NOT EXISTS (
            SELECT 1
            FROM {v_source} s
            WHERE s.{v_pk} = t.{v_pk}
          )
        """

    with conn.cursor() as cursor:
        cursor.execute(v_sql)
        v_deletes = cursor.rowcount

    print(f"❄ Deletes uitgevoerd met strategie {v_delete_strategy}. Aantal deletes: {v_deletes}")

    conn.commit()
    return v_deletes

### Logging

In [143]:

# with conn.cursor() as cursor:
#     cursor.execute(
#         """
#         INSERT INTO LOGGING.RUN_ENTITY_LOG (
#             RUN_ID,
#             START_TS,
#             END_TS,
#             ENTITY_NAME,
#             INSERT_COUNT,
#             UPDATE_COUNT,
#             DELETE_COUNT,
#             UNCHANGED_COUNT,
#             DUPLICATE_INSERT_COUNT,
#             DUPLICATE_UPDATE_COUNT,
#             KEY_ERROR_COUNT
#         )
#         VALUES (
#             %s, %s, CURRENT_TIMESTAMP(), %s, %s, %s, %s, %s, %s, %s, %s
#         )
#         """,
#         (
#             run_id,
#             v_start_ts,
#             v_entity,
#             v_inserts,
#             v_updates,
#             v_deletes,
#             v_unchanged,
#             v_duplicate_inserts,
#             v_duplicate_updates,
#             v_key_errors,
#         ),
#     )

# conn.commit()

### Uitvoeren van het proces

In [144]:
# Script om het CDC proces uit te voeren en te loggen in de database
with conn.cursor() as cursor:
    cursor.execute("SELECT COALESCE(MAX(RUN_ID), 0) + 1 AS NEXT_RUN_ID FROM LOGGING.RUN_LOG")
    run_id = cursor.fetchone()[0]

print(f"❄ Starten CDC run met RUN_ID={run_id}...")

with conn.cursor() as cursor:
    cursor.execute(
        """
        INSERT INTO LOGGING.RUN_LOG (RUN_ID, START_TS, STATUS)
        VALUES (%s, CURRENT_TIMESTAMP(), 'RUNNING')
        """,
        (run_id,),
    )
conn.commit()

print(f"❄ CDC run gestart met RUN_ID={run_id}.")

fatal_error = False

with conn.cursor() as cursor:
    cursor.execute(
        """
        SELECT CONFIG_ID
        FROM CDC.CDC_CONFIG
        WHERE IS_ACTIVE = TRUE
        ORDER BY CONFIG_ID
        """
    )
    config_ids = [row[0] for row in cursor.fetchall()]

    if not config_ids:
        print("❄ Geen (actieve) configuraties gevonden in CDC_CONFIG. CDC run stopt...")
        fatal_error = True

for config_id in config_ids:
    try:
        v_start_ts = None

        with conn.cursor() as cursor:
            cursor.execute("SELECT CURRENT_TIMESTAMP()")
            v_start_ts = cursor.fetchone()[0]

        init_values = initialise(config_id, run_id)
        if not isinstance(init_values, dict):
            print(f"❄ Initialiseren van entity met CONFIG_ID={config_id} mislukt. Fout: {init_values}. Entity wordt overgeslagen. Beginnen volgende entity...")
            continue

        v_entity = init_values["v_entity"]
        v_source = init_values["v_source"]
        v_target = init_values["v_target"]
        v_pk = init_values["v_pk"]

        error_values = detect_errors(run_id, init_values)
        v_duplicate_inserts = error_values["v_duplicate_inserts"]
        v_duplicate_updates = error_values["v_duplicate_updates"]
        # v_unchanged = error_values["v_unchanged"]
        v_key_errors = error_values["v_key_errors"]

        v_inserts = execute_inserts(run_id, init_values)
        v_updates = execute_updates(run_id, init_values)
        v_deletes = execute_deletes(run_id, init_values)

        # Ongewijzigde rijen: actieve target rijen met dezelfde hash als source

        with conn.cursor() as cursor:
            cursor.execute(
                """
                INSERT INTO LOGGING.RUN_ENTITY_LOG (RUN_ID, START_TS, END_TS, ENTITY_NAME, ROWS_INSERTED, ROWS_UPDATED, ROWS_DELETED, ROWS_UNCHANGED, DUPLICATE_INSERTS, DUPLICATE_UPDATES, KEY_ERRORS)
                VALUES (%s, %s, CURRENT_TIMESTAMP(), %s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (run_id, v_start_ts, v_entity, v_inserts, v_updates, v_deletes, v_unchanged, v_duplicate_inserts, v_duplicate_updates, v_key_errors),
            )
        conn.commit()

        print(f"❄ Entity {v_entity} verwerkt. Inserts: {v_inserts}, Updates: {v_updates}, Deletes: {v_deletes}, Ongewijzigd: {v_unchanged}, Duplicate Inserts: {v_duplicate_inserts}, Duplicate Updates: {v_duplicate_updates}, Key Errors: {v_key_errors}")

    except Exception as e:
        print("❄ Rollback uitvoeren vanwege fout...")
        conn.rollback()
        fatal_error = True

        with conn.cursor() as cursor:
            cursor.execute(
                """
                INSERT INTO LOGGING.RUN_ERROR_LOG (RUN_ID, ENTITY_NAME, ERROR_ROW)
                VALUES (%s, %s, OBJECT_CONSTRUCT('message', %s, 'config_id', %s))
                """,
                (run_id, v_entity, str(e), config_id),
            )
        conn.commit()

        print(f"❄ Fout(en) gevonden bij verwerken van entity {v_entity} (CONFIG_ID={config_id}): {str(e)}")

with conn.cursor() as cursor:
    print("❄ CDC run afronden...")
    cursor.execute(
        """
        UPDATE LOGGING.RUN_LOG
        SET END_TS = CURRENT_TIMESTAMP(),
            STATUS = %s
        WHERE RUN_ID = %s
        """,
        ('FAILED' if fatal_error else 'COMPLETED', run_id),
    )
conn.commit()

print(f"❄ CDC run afgerond. RUN_ID={run_id}, STATUS={'FAILED' if fatal_error else 'COMPLETED'}")

❄ Starten CDC run met RUN_ID=51...
❄ CDC run gestart met RUN_ID=51.
❄ Beginnen met verwerken van entity Employee...
❄ Duplicate inserts detecteren...
❄ Duplicate inserts gedetecteerd: 0
❄ Duplicate updates detecteren...
❄ Duplicate updates gedetecteerd: 0
❄ Key errors detecteren...
❄ Key errors gedetecteerd: 286
❄ Inserts uitvoeren...
❄ Inserts uitgevoerd. Aantal inserts: 0
❄ Updates uitvoeren...
❄ Updates uitgevoerd met strategie HISTORY. Aantal updates: 0
❄ Deletes uitvoeren...
❄ Deletes uitgevoerd met strategie SOFT. Aantal deletes: 0
❄ Entity Employee verwerkt. Inserts: 0, Updates: 0, Deletes: 0, Ongewijzigd: 1, Duplicate Inserts: 0, Duplicate Updates: 0, Key Errors: 286
❄ Beginnen met verwerken van entity Customer...
❄ Duplicate inserts detecteren...
❄ Duplicate inserts gedetecteerd: 0
❄ Duplicate updates detecteren...
❄ Duplicate updates gedetecteerd: 0
❄ Key errors detecteren...
❄ Key errors gedetecteerd: 0
❄ Inserts uitvoeren...
❄ Inserts uitgevoerd. Aantal inserts: 0
❄ Updates